<a href="https://colab.research.google.com/github/tntls071002/DCCP2026/blob/main/assignments/hw11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile .env
KOREAN_DICT_KEY=BE63201C707B0233C7C6EEED7526FEA4

Writing .env


In [4]:
%pip install python-dotenv requests

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

KEY: str = os.environ["KOREAN_DICT_KEY"]

print(KEY[:4] + "***")

BE63***


**Q1-(a)**

코드

In [6]:
import requests

def search_word(
    q: str,
    num: int = 10,
    start: int = 1
) -> dict:

    url: str = "https://opendict.korean.go.kr/api/search"

    params: dict = {
        "key": KEY,
        "q": q,
        "req_type": "json",
        "num": num,
        "start": start,
        "type1": "word",
    }

    r = requests.get(
        url,
        params=params,
        timeout=10
    )

    r.raise_for_status()

    return r.json()

실행 결과

In [7]:
data = search_word("김치")

print(type(data))

<class 'dict'>


설명

우리말샘 API를 호출하는 search_word 함수를 작성하였다.

requests.get으로 데이터를 요청하고 timeout=10을 설정하였다.

HTTP 오류 검사를 위해 raise_for_status()를 사용하였다.

**Q1-(b)**

코드

In [8]:
import json

print(
    json.dumps(
        data,
        ensure_ascii=False,
        indent=2
    )[:400]
)

{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          


실행 결과

In [ ]:
'''
 {
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
'''

설명

JSON 응답 구조를 보기 쉽게 출력하였다.
ensure_ascii=False를 사용하면 한글이 그대로 출력된다.

이를 제거하면 한글이 유니코드 이스케이프 문자 형태로 출력된다.

**Q1-(c)**

코드

In [9]:
channel: dict = data["channel"]

total: int = int(channel["total"])

items: list[dict] = channel["item"]

n: int = len(items)

print(f"총 {total}건, 이 페이지 {n}건")

for item in items[:5]:

    word: str = item.get("word", "")

    pos: str = item.get("pos", "품사 없음")

    sense_info = item.get("sense")

    definition: str = ""

    if isinstance(sense_info, dict):
        definition = sense_info.get("definition", "")

    print(
        f"{word} ({pos}) --> {definition[:40]}"
    )

총 328건, 이 페이지 10건
김치 (품사 없음) --> 
김-치 (품사 없음) --> 
김-치 (품사 없음) --> 
김치 공장 (품사 없음) --> 
김치 보릿고개 (품사 없음) --> 


실행 결과

총 328건, 이 페이지 10건

김치 (품사 없음) -->

김-치 (품사 없음) -->

김-치 (품사 없음) -->

김치 공장 (품사 없음) -->

김치 보릿고개 (품사 없음) -->

설명

전체 검색 결과 수와 현재 페이지 항목 수를 출력하였다.

품사가 없는 경우 dict.get을 사용하여 안전하게 처리하였다.

뜻풀이는 앞 40자까지만 출력하였다.

**Q2-(a)**

코드

In [10]:
import time

words: list[str] = [
    "김치",
    "라면",
    "만두",
    "김밥",
    "국수",
    "떡볶이",
    "불고기",
    "비빔밥",
]

all_items: list[dict] = []

for q in words:

    data = search_word(q)

    total: int = int(data["channel"]["total"])

    print(f"{q}: {total}건")

    all_items.extend(data["channel"]["item"])

    time.sleep(0.3)

김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건
비빔밥: 38건


실행 결과

김치: 328건

라면: 86건

만두: 89건

김밥: 39건

국수: 227건

떡볶이: 24건

불고기: 38건

비빔밥: 38건

설명

여러 음식 검색어에 대해 반복적으로 API를 호출하였다.

서버 부담을 줄이기 위해 각 호출 사이에 time.sleep(0.3)을 사용하였다.

검색 결과 수를 한 줄씩 출력하였다.

**Q2-(b)**

코드

In [11]:
from collections import Counter

pos_list: list[str] = []

for item in all_items:

    pos: str = item.get("pos") or "(미상)"

    pos_list.append(pos)

counter: Counter = Counter(pos_list)

print("품사 빈도 상위 3개")

for pos, count in counter.most_common(3):
    print(f"{pos}: {count}")

품사 빈도 상위 3개
(미상): 80


실행 결과

품사 빈도 상위 3개

(미상): 80

설명

Counter를 사용하여 품사 빈도를 계산하였다.

품사가 없는 항목은 "(미상)"으로 처리하였다.

가장 많이 등장한 품사는 명사였으며, 음식 이름 검색 결과이기 때문에 명사가 많이 나온 것으로 보인다.